### This notebook computes the "VSB10_A. Availability of Water by Basin for the Agricultural Sector" indicator for the 27 basins of IKI Project

**Spanish:** Disponibilidad de agua por cuenca para el sector agrario

**Created:** 1/22/2026 by Sophia Bakar (sbakar@rti.org)

**Project #:** 0219481  

**Last modified:** 

**Status:** in progress

**QA Status:** reviewed by  
 
**Inputs:**   

**Outputs:** 
 
**Assumptions:** 
 
**Future work:** 
 
**Notes:** 

In [76]:
import pandas as pd
import numpy as np
import geopandas as gpd
import sqlite3
import matplotlib.pyplot as plt
import re 

#### Seleccion de rutas como funcion del usuario

In [77]:
# set up user and database path
#user = 'jmayo'
#user= 'cpickering'
#user = 'sgilson'
#user = 'nreynolds'
#user = 'sbakar'
entry_folder= "" #"General\\"
user = 'etriana'

entry_path = fr"C:\Users\{user}\Research Triangle Institute\IKI Peru Project - {entry_folder}Interno"

db_path = fr"{entry_path}\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"

wateralloc_db = fr"{entry_path}\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"

# subbasins_shapefile = fr"{entry_path}\AI2b_Modelacion\Grupos_Modelacion\GIS_WaterALLOC_General\Peru_AHD_with_districts.shp"


In [78]:
# set up indicator ID and get scenarios from database
IndID= "310" #Indicator ID (Exposure = 2 + 0X where X is the Exposure Indicator number, Peligro= 1 +0x, VSB= 3 +0x, VSS= 4 +0x, VCA= 5 +0x)
conn = sqlite3.connect(db_path)

scenarios_df = pd.read_sql_query(
    """
    SELECT ScnID, ScnName
    FROM ScnMod
    """,
    conn
)

conn.close()

# For now: only baseline and first future
#scenario_ids = scenarios_df.loc[
#    scenarios_df['ScnID'].isin([1, 2]), 'ScnID'
#].tolist()

# for all scenarios:
scenario_ids = scenarios_df['ScnID'].tolist()

scenarios_df

,ScnID,ScnName
0,1,Linea Base
1,2,CC CMIP6 85


In [79]:
## check what scenarios are available in the WaterALLOC database
# Connect to WaterALLOC database
conn_wa = sqlite3.connect(wateralloc_db)

# Query available scenarios
scenarios_query = """
SELECT WaScnID, Scenario, IndScnName
FROM Scenarios
ORDER BY Scenario
"""

wa_scenarios_df = pd.read_sql_query(scenarios_query, conn_wa)

print("Available scenarios in WaterALLOC DB:")
for s in wa_scenarios_df["Scenario"]:
    print(f" - {s}")
           
conn_wa.close()


Available scenarios in WaterALLOC DB:
 - CC_CMIP6_85_2050
 - Linea_Base_2020
 - Linea_Base_2020_Embalses


#### Find errors matching WaterALLOC scenarios to the indicators

In [80]:
# Find issue with not matching scenario names with the indicator database
# if IndScnName is null, print an error message
mismatched_scenarios = wa_scenarios_df[wa_scenarios_df["IndScnName"].isnull()]
if not mismatched_scenarios.empty:
    print("\nError: The following WaterALLOC scenarios do not have matching indicator scenario names:")
    for index, row in mismatched_scenarios.iterrows():
        print(f" - WaScnID: {row['WaScnID']}, Scenario: {row['Scenario']}")
        
# if the names in IndScnName do not match any of the ScnName in scenarios_df, print an error message
for index, row in wa_scenarios_df.iterrows():
    if pd.notnull(row["IndScnName"]):
        if row["IndScnName"] not in scenarios_df["ScnName"].values:
            print(f"\nError: WaterALLOC scenario '{row['IndScnName']}' does not match any ScnName in indicator database.")  
            

### Process all the WaterALLOC Scenarios
Los escenarios en WaterALLOC deben corresponder a un escenario en la base de datos de indicadores. Es esta seccion procesamos todos los escenarios en la base de datos vinculando con el escenario correspondiente en la base de datos de indicadores.

Este calculo necesita un indicador unico en la base de datos de indicadores para las combinaciones de  escenarios de indicadores y WaterALLOC.  
#### Method
Using SQL we attach the indicators database and execute an insert query in the indicators database using the processed data from the WaterALLOC scenarios database.  

Only the COMIDs processed to the WaterALLOC database are available.  The risk calculation should handle the missing COMID values.

The indicator for each COMID is calculated as the average total water available at the COMID, which includes the local available and the upstream available water minus the local demand.  

In [81]:
# Connect to WaterALLOC database
conn_wa = sqlite3.connect(wateralloc_db)
cursor = conn_wa.cursor()
# Attach RiesgoDB database
attach_query = fr"ATTACH DATABASE '{db_path}' AS RiesgoDB;"
cursor.execute(attach_query)

availability_all = []

## Crear un loop por cada escenario en el WaterALLOC DB 
# faster: iterate rows as namedtuples
for row in wa_scenarios_df.itertuples(index=True, name="Row"):
    waScn_ID = row.WaScnID
    print(f"\nProcessing WA scenario: {row.Scenario} (ScnID={row.WaScnID}) with indicator scenario name: {row.IndScnName}")
    
    scenario_name = scenarios_df.loc[
        scenarios_df['ScnName'] == row.IndScnName 
    ] 
    
    ind_ScnID = scenario_name.ScnID.values[0]
    print(f"\tLinked with scenario: {scenario_name.ScnName.values[0]} (ScnID={ind_ScnID})")
    
    # Delete previous entries for this indicator
    delete_query = fr"DELETE FROM [RiesgoDB].IndValues_WaALLOC WHERE WaScnID = {waScn_ID};"
    cursor.execute(delete_query)

    
    query_avaltotal = fr"""
    INSERT INTO [RiesgoDB].IndValues_WaALLOC
        SELECT {waScn_ID} AS WaScnID, {ind_ScnID} as IndID, b.comid AS COMID, 
            avg(b.[Oferta Local Sup]+b.[Oferta Entrada]-b.[Dem Local Sup]) AS [Value] 
        FROM [WAMSS_Balance por COMID (+Indice de estres)] AS b
        JOIN
            (SELECT RunID
            FROM WAMMS_RunsInfo
            Where WaScnID = {waScn_ID}) AS r 
            ON r.RunID = b.RunID
        WHERE comid not NULL
        GROUP BY b.comid;
    """
    # execute insert query and print the number of inserted rows
    rows_inserted = cursor.execute(query_avaltotal).rowcount
    print(f"\tInserted {rows_inserted} rows into IndValues_WaALLOC table.")
    print("\tInserted values into IndValues_WaALLOC table.")
       
    
# Commit and close DB connection
conn_wa.commit()
conn_wa.close() 





Processing WA scenario: CC_CMIP6_85_2050 (ScnID=1) with indicator scenario name: CC CMIP6 85
	Linked with scenario: CC CMIP6 85 (ScnID=2)
	Inserted 543 rows into IndValues_WaALLOC table.
	Inserted values into IndValues_WaALLOC table.

Processing WA scenario: Linea_Base_2020 (ScnID=3) with indicator scenario name: Linea Base
	Linked with scenario: Linea Base (ScnID=1)
	Inserted 543 rows into IndValues_WaALLOC table.
	Inserted values into IndValues_WaALLOC table.

Processing WA scenario: Linea_Base_2020_Embalses (ScnID=4) with indicator scenario name: Linea Base
	Linked with scenario: Linea Base (ScnID=1)
	Inserted 295 rows into IndValues_WaALLOC table.
	Inserted values into IndValues_WaALLOC table.
